# 加州房价进阶:5 个让模型变强的工程技巧

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/california_housing_advanced.ipynb)

本 notebook 是 [`california_housing_tutorial.ipynb`](california_housing_tutorial.ipynb) 的**进阶续篇**。
基础教程教你**走完一遍机器学习流程**;这一份教你**让分数变得更好**——也就是真实工作中花时间最多的部分。

> 🎯 **写给谁看**:刚学完基础教程,想知道「下一步还能做什么」的同学。
> **不需要决策树背景**——本 notebook 第 2 节会用 5 分钟带你完全搞懂决策树。

## 你将学到 5 个工程技巧

| # | 技巧 | 解决的问题 |
| --- | --- | --- |
| 1 | **目标变换 `log1p`** | 房价分布严重右偏,取对数后更对称 |
| 2 | **去掉截断样本** | `MedHouseVal == 5.0` 是人为天花板,会污染模型 |
| 3 | **比率特征工程** | 「人均房间数」比「平均房间数 + 平均人数」更直接 |
| 4 | **海岸线距离** | 加州沿海贵、内陆便宜,显式给模型这个特征 |
| 5 | **现代 GBM 三剑客** | XGBoost / LightGBM / CatBoost——表格数据冠军模型 |

每一节都遵循同一个三段式:
1. **直觉**:用一句大白话讲清为什么要做这个;
2. **代码**:最小可运行实现;
3. **效果**:训练一遍,看测试 RMSE 改善了多少。

最后我们把所有技巧叠在一起,看看从基础教程的 RMSE 0.471 能压到多少。

## 目录
0. 阅读须知
1. 环境与数据加载
2. **决策树 5 分钟入门**(没学过的同学请细读)
3. 集成树:森林与提升
4. 建立基线分数
5. 技巧 1:目标变换 `log1p`
6. 技巧 2:排除截断样本
7. 技巧 3:比率特征工程
8. 技巧 4:海岸线距离
9. 技巧 5:XGBoost / LightGBM / CatBoost
10. 综合实验:把所有技巧叠在一起
11. 总结与下一步学习路径


## 0. 阅读须知

> 📘 **小知识:什么叫「特征工程」?**
> 「特征」(feature)就是给模型看的输入列。「特征工程」就是**人为设计更有信息量的列**——
> 比如把「房间数」和「人数」合并成「人均房间数」。
> 在表格数据上,**好的特征工程往往比换更复杂的模型更有效**。

> 📘 **小知识:为什么一直在说「RMSE」?**
> RMSE = Root Mean Squared Error,翻译成人话就是**「平均预测误差」**。
> 比如 RMSE = 0.5 意味着我们对加州街区房价的预测平均偏差约 \$50,000(数据单位是 \$100K)。
> **越小越好。**

本 notebook 假设你**装了**(Kaggle 默认环境全部内置):
- `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`
- `xgboost`, `lightgbm`, `catboost` ← 第 9 节才用到

如果本地缺少其中之一,在最上面跑:
```python
!pip install xgboost lightgbm catboost
```


## 1. 环境与数据加载


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

bunch = fetch_california_housing(as_frame=True)
df = bunch.frame
print(f'数据形状: {df.shape}')
df.head()


In [ ]:
# 训练 / 测试划分,保持与基础教程一致
TARGET = 'MedHouseVal'
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')


## 2. 决策树 5 分钟入门

如果你**没学过决策树**,这一节是核心。我们后面用到的 GBM、XGBoost、LightGBM、CatBoost
**全部都是树的延伸**——理解一棵树,后面所有算法的内部细节就一通百通。

### 2.1 用「20 个问题」类比

你在玩「20 个问题」游戏,目标是猜出对方心里的动物。最好的提问策略是:
- **第一问要切分得最干净**:比如问「是不是哺乳动物?」直接把候选切成两半;
- **每一问都基于前一问的答案**:之前回答「是」之后,下一问会更细,比如「会飞吗?」
- **问到答案足够确定就停止**

**决策树就是把这套策略自动化**:模型会自动找出**最有信息量的提问顺序**,
直到把样本分成「同质」的小组。每个小组给一个预测值。


### 2.2 真的画一棵树看看

我们对加州数据训练**一棵小决策树**(只允许深度 3,这样能画下来),然后可视化它。


In [ ]:
tiny_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tiny_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tiny_tree,
          feature_names=feature_cols,
          filled=True, rounded=True,
          impurity=False, fontsize=10, ax=ax)
plt.title('A tiny decision tree (depth=3) on California Housing', fontsize=13)
plt.tight_layout()
plt.show()

print(f'测试 RMSE: {np.sqrt(mean_squared_error(y_test, tiny_tree.predict(X_test))):.3f}')


### 2.3 怎么读这张图?

每个**方框**是一个「提问节点」。例如根节点写着 `MedInc <= 5.04`:
> 「这个街区的家庭收入中位数(单位万)是不是小于等于 5.04?」

- 答 **是** → 走左边分支(继续提问)
- 答 **否** → 走右边分支(继续提问)

到达**叶子节点**(没有再分支的方框)时,框里的 `value = X.X` 就是这个叶子的预测房价。
**对一个新样本预测**,就是从根节点开始按问答走到叶子,把叶子的 value 报出来。

> 📘 **关键直觉**:决策树**不假设线性关系**,它本质上是「分段常数函数」——把特征空间切成
> 一格一格的小区块,每个小区块给一个固定预测值。
> 这正是树模型在加州数据上能秒杀线性回归的原因:**经纬度 vs 房价根本不是线性的**。

> 📘 **小知识:树是怎么选问题的?**
> 对回归问题,树会枚举每个特征 × 每个可能的切分点,选**让两边方差总和最小**的那个。
> 直觉:切完之后两组样本的房价越「集中」越好。这个准则叫 MSE/方差缩减。


### 2.4 单棵树的问题

```text
            缺点                    后果
  ─────────────────────────  ───────────────────────────
  容易过拟合(死记训练样本)   训练 RMSE 很低,测试 RMSE 飙高
  对数据微小扰动很敏感         换一组训练样本可能长出完全不同的树
  预测是阶梯状的               边界突变,光滑度差
```

这就引出了**集成树**——用一群树代替一棵树。


## 3. 集成树:森林与提升

集成(ensemble)= 把多个弱模型组合成一个强模型。在表格数据上,集成树是**事实上的冠军方法**。

### 3.1 随机森林(Bagging,「群体投票」)

直觉:**100 棵互不相同的树,各预测一个值,取平均**。

怎么做到「互不相同」?
- 每棵树用**训练集的随机子集**(bootstrap 采样);
- 每次切分**只看随机子集的特征**。

> 📘 **为什么这样会变强?**
> 单棵树容易乱猜,但 100 棵树一起乱猜的**平均**会更稳——这是统计学的**大数定律**。
> 用一个比喻:一个人估房价可能离谱,100 个独立的人估完取中位数会非常准。

### 3.2 梯度提升(Boosting,「接力修补」)

直觉:**第 1 棵树先做个粗糙预测;第 2 棵树专门学习第 1 棵的错误;第 3 棵学习第 1+2 之后的剩余错误...**
依次接力,每一棵都在「补丁」前面所有树的偏差。

最终预测 = 所有树的预测之和(乘以一个小学习率,避免迈步太大)。

> 📘 **为什么这样会变强?**
> Bagging 解决「方差大」(模型乱跳);Boosting 解决「偏差大」(模型不够细)。
> Boosting 通常**精度更高**,但**更容易过拟合**——所以要用学习率、树数、树深来正则化。


## 4. 建立基线分数

在引入任何技巧前,先记下一个**基线模型**的成绩——后面所有改进都要和它比。
我们用一个轻量的 `GradientBoostingRegressor`(默认参数)。


In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name, target_log=False):
    """训练 + 预测 + 报告 RMSE / MAE / R²。
    target_log=True 时,假设模型对 log1p(y) 训练,我们要把预测 expm1 还原。"""
    if target_log:
        model.fit(X_tr, np.log1p(y_tr))
        pred_te = np.expm1(model.predict(X_te))
    else:
        model.fit(X_tr, y_tr)
        pred_te = model.predict(X_te)

    rmse = np.sqrt(mean_squared_error(y_te, pred_te))
    mae  = mean_absolute_error(y_te, pred_te)
    r2   = r2_score(y_te, pred_te)
    print(f'{name:<35} | Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}')
    return {'name': name, 'rmse': rmse, 'mae': mae, 'r2': r2, 'fitted': model}


results = []
baseline = evaluate(GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                              learning_rate=0.1, random_state=42),
                    X_train, y_train, X_test, y_test, '0. Baseline GBR')
results.append(baseline)


**基线 RMSE 记下来:接下来每个技巧都和它比。**


## 5. 技巧 1:目标变换 `log1p`

### 5.1 直觉

加州房价是**严重右偏**的:大多数街区便宜,少数街区贵到离谱。
对模型来说,这种分布有个麻烦——**贵房子的预测误差(以美元算)会比便宜房子大得多**,
拉高整体 RMSE,模型为了「迁就贵房子」会牺牲便宜房子的精度。

**解法**:对目标取对数。`log1p(x) = log(1 + x)`(用 1+x 是为了能处理 x=0)。
对数会**压缩大值、放大小值**,让分布变对称。

> 📘 **为什么对数能这么做?**
> `log(10) - log(1) = log(100) - log(10) = log(1000) - log(100) ≈ 2.3`。
> 也就是说,**乘法关系在对数空间变成加法关系**。
> 房价从 10 万翻到 100 万,在原空间是 90 万的差距,在对数空间是固定的 2.3。模型更好处理。

### 5.2 代码


In [ ]:
# 看看变换前后的分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Original: MedHouseVal')
axes[0].set_xlabel('MedHouseVal')

axes[1].hist(np.log1p(y_train), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('After log1p: log(1 + MedHouseVal)')
axes[1].set_xlabel('log1p(MedHouseVal)')

plt.tight_layout()
plt.show()

print(f'原始分布偏度:    {pd.Series(y_train).skew():.3f}')
print(f'log1p 后偏度:    {pd.Series(np.log1p(y_train)).skew():.3f}')


可以看到 log1p 后右尾被压扁,分布近似对称(偏度从约 1.0 降到 ~0)。

### 5.3 训练 + 对比


In [ ]:
r = evaluate(GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.1, random_state=42),
             X_train, y_train, X_test, y_test,
             '1. + log1p target', target_log=True)
results.append(r)


> ⚠️ **注意**:`log1p` 会让模型在**对数空间**做训练,所以预测后必须用 `expm1` 反变换回原空间。
> `evaluate` 函数已经帮你做了这件事。


## 6. 技巧 2:排除截断样本

### 6.1 直觉

记不记得基础教程里那条吐槽——`MedHouseVal == 5.0` 处有大量样本堆积?
这是当年人口普查时**人为地把 \$500,000 以上的房价都记成 5.0**(隐私保护)。

对模型来说,这是一个**学错的天花板**:它学到「最贵也就 5.0 了」,但实际真实房价可能远高于此。
**简单粗暴的解法**:训练时直接丢掉这些样本,告诉模型「别管 5.0 那个砖墙」。

> 📘 **小知识:什么叫数据「截断」(censored)?**
> 真实值因为某种原因被压在某个范围内(隐私、传感器上限、问卷选项最大值)。
> 截断会让模型对边界附近的预测产生**系统性偏差**。统计学里专门有「Tobit 模型」处理这种情况。

### 6.2 代码


In [ ]:
mask_train = y_train < 5.0
print(f'去掉 {(~mask_train).sum()} 个截断样本,占训练集 {(~mask_train).mean()*100:.1f}%')

X_train_drop = X_train[mask_train]
y_train_drop = y_train[mask_train]

r = evaluate(GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.1, random_state=42),
             X_train_drop, y_train_drop, X_test, y_test,
             '2. + drop capped (y<5.0)')
results.append(r)


> ⚠️ **测试集仍然包含截断样本**——评估必须在「真实分布」上做,不能为了好看而双标。


## 7. 技巧 3:比率特征工程

### 7.1 直觉

原始数据里有几个特征其实**不太合理**:
- `AveRooms`:平均每户房间数(但有些街区是 5 间小公寓平均,有些是 5 间大别墅平均);
- `AveOccup`:平均每户人数(同样问题)。

工程师常用的招:**把它们除一除,得到更直接的语义**。

| 新特征 | 含义 | 直觉 |
| --- | --- | --- |
| `rooms_per_person` = `AveRooms / AveOccup` | 人均房间数 | 越大越宽敞,越贵 |
| `bedrooms_per_room` = `AveBedrms / AveRooms` | 卧室占比 | 越小说明厨房/客厅越多,房子越「正经」 |
| `pop_per_household` = `Population / AveOccup` | 户数 | 与街区规模有关 |

> 📘 **为什么不能直接让模型自己学?**
> 树模型理论上能学出 `A / B` 的近似(只要切的够细),但**显式给出比率会大大降低需要的树深度**,
> 训练更快、更稳。线性模型则**完全无法**学出比率(它只会加减)——所以特征工程对线性模型至关重要。

### 7.2 代码


In [ ]:
def add_ratio_features(X):
    X = X.copy()
    X['rooms_per_person']   = X['AveRooms']  / X['AveOccup']
    X['bedrooms_per_room']  = X['AveBedrms'] / X['AveRooms']
    X['pop_per_household']  = X['Population'] / X['AveOccup']
    return X

X_train_ratio = add_ratio_features(X_train)
X_test_ratio  = add_ratio_features(X_test)

print(f'原始特征数: {X_train.shape[1]} → 工程化后: {X_train_ratio.shape[1]}')
X_train_ratio.head(3)


In [ ]:
r = evaluate(GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.1, random_state=42),
             X_train_ratio, y_train, X_test_ratio, y_test,
             '3. + ratio features')
results.append(r)


## 8. 技巧 4:海岸线距离

### 8.1 直觉

地图上沿海一条线明显是高价区。但模型只看到 `Latitude` 和 `Longitude`——
它得**自己学会沿海**,这要花很多树深度。

我们可以**直接告诉模型每个点离最近海岸线有多远**,把这个非线性结构直接喂给它。

### 8.2 海岸线坐标 + 距离计算

我们用 13 个粗略坐标点描述加州海岸线(从南到北),计算每个街区到这条折线最近点的**球面距离**(单位:公里)。


In [ ]:
# California coastline: rough polyline (latitude, longitude), south to north
COASTLINE = np.array([
    (32.55, -117.06),  # Imperial Beach (near San Diego)
    (33.00, -117.27),  # Oceanside
    (33.55, -118.00),  # Long Beach
    (33.95, -118.40),  # near LAX
    (34.40, -119.69),  # Santa Barbara
    (35.00, -120.62),  # Pismo Beach
    (35.60, -121.10),  # Cambria
    (36.50, -121.95),  # Monterey
    (37.50, -122.50),  # San Francisco
    (38.30, -123.00),  # north of SF
    (39.30, -123.80),  # Mendocino
    (40.50, -124.40),  # near Eureka
    (41.75, -124.20),  # Crescent City
])

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def coast_distance_km(lat, lon, coast=COASTLINE):
    """For each (lat, lon) row, compute min haversine distance to any coast point."""
    lat = np.asarray(lat).reshape(-1, 1)
    lon = np.asarray(lon).reshape(-1, 1)
    cl_lat = coast[:, 0].reshape(1, -1)
    cl_lon = coast[:, 1].reshape(1, -1)
    return haversine_km(lat, lon, cl_lat, cl_lon).min(axis=1)


coast_dist_train = coast_distance_km(X_train['Latitude'].values, X_train['Longitude'].values)
coast_dist_test  = coast_distance_km(X_test['Latitude'].values,  X_test['Longitude'].values)
coast_dist_all   = coast_distance_km(df['Latitude'].values,      df['Longitude'].values)

print(f'到海岸线距离统计 (km): min={coast_dist_all.min():.1f}, '
      f'median={np.median(coast_dist_all):.1f}, max={coast_dist_all.max():.1f}')


### 8.3 看一眼新特征

把每个街区按经纬度画在图上,颜色编码「到海岸距离」,验证我们的距离场是合理的。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sc = axes[0].scatter(df['Longitude'], df['Latitude'],
                     c=coast_dist_all, cmap='RdYlBu_r',
                     s=4, alpha=0.5)
axes[0].plot(COASTLINE[:, 1], COASTLINE[:, 0], 'k-o', lw=1.5, ms=4, label='Coastline approx')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title('Distance to nearest coastline (km)')
axes[0].legend()
plt.colorbar(sc, ax=axes[0], label='Distance (km)')

axes[1].scatter(coast_dist_all, df['MedHouseVal'], alpha=0.15, s=8, color='steelblue')
axes[1].set_xlabel('Distance to coast (km)')
axes[1].set_ylabel('MedHouseVal ($100K)')
axes[1].set_title('Coast distance vs House Price')

plt.tight_layout()
plt.show()


**右图**清晰显示:**离海越远,房价越低**,这正是我们想给模型看到的信号。

### 8.4 训练 + 对比


In [ ]:
X_train_coast = X_train_ratio.copy()
X_test_coast  = X_test_ratio.copy()
X_train_coast['coast_dist_km'] = coast_dist_train
X_test_coast['coast_dist_km']  = coast_dist_test

r = evaluate(GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.1, random_state=42),
             X_train_coast, y_train, X_test_coast, y_test,
             '4. + coastline distance')
results.append(r)


## 9. 技巧 5:现代 GBM 三剑客

`GradientBoostingRegressor` 是 sklearn 自带的「教科书级」实现——清晰、稳定、但**慢**。
工业界与 Kaggle 上人人在用的是三个开源库:

| 库 | 出身 | 特色 |
| --- | --- | --- |
| **XGBoost** | 陈天奇 (2014) | 第一个把 GBM 工程化到极致,正则化设计完整 |
| **LightGBM** | 微软 (2016) | 直方图算法 + leaf-wise 生长,速度上数量级领先 |
| **CatBoost** | Yandex (2017) | 自带类别特征处理,默认参数下泛化更好 |

> 📘 **为什么这三个比 sklearn 快这么多?**
> sklearn 的 GBR 每次切分要扫描所有连续值;XGBoost 用了**预排序优化**,
> LightGBM 用了**直方图分箱**(把连续特征压缩成 256 个桶)。
> 在 20K 行数据上,差距还不夸张;到了百万行,LightGBM 比 sklearn 快 50–100 倍。

### 9.1 三家上场


In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# 我们继续用「ratio + coast」工程化后的数据集
X_tr = X_train_coast
X_te = X_test_coast

xgb_model = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0)
r = evaluate(xgb_model, X_tr, y_train, X_te, y_test, '5a. XGBoost')
results.append(r)

lgb_model = lgb.LGBMRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=-1, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1)
r = evaluate(lgb_model, X_tr, y_train, X_te, y_test, '5b. LightGBM')
results.append(r)

cat_model = CatBoostRegressor(
    iterations=400, learning_rate=0.05, depth=6,
    random_state=42, verbose=0)
r = evaluate(cat_model, X_tr, y_train, X_te, y_test, '5c. CatBoost')
results.append(r)


> 📘 **小知识:`subsample` 和 `colsample_bytree` 是什么?**
> 这是**随机性正则化**:每棵树只用 80% 的样本(`subsample`)和 80% 的特征(`colsample_bytree`)。
> 类似 Bagging 的思想,让每棵树各有侧重,集成时减少过拟合。


## 10. 综合实验:把所有改进叠在一起

最后,我们把**前 4 个技巧 + LightGBM**(目前看似最快/最准)叠加,看看最终能跑出什么分数。


In [ ]:
# log1p target + drop capped + ratio + coast + LightGBM
mask_train = y_train < 5.0
X_tr_full = X_train_coast[mask_train]
y_tr_full = y_train[mask_train]

final = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1)

r = evaluate(final, X_tr_full, y_tr_full, X_test_coast, y_test,
             '6. ALL: log1p + drop + ratio + coast + LGBM',
             target_log=True)
results.append(r)


### 改进汇总


In [ ]:
leaderboard = pd.DataFrame(results)[['name', 'rmse', 'mae', 'r2']].copy()
leaderboard['Δ_RMSE_vs_baseline'] = leaderboard['rmse'] - leaderboard.iloc[0]['rmse']
leaderboard['rel_improvement_%'] = (
    (leaderboard.iloc[0]['rmse'] - leaderboard['rmse']) / leaderboard.iloc[0]['rmse'] * 100
)
leaderboard.round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#999999'] + ['#1f77b4'] * 4 + ['#ff7f0e'] * 3 + ['#d62728']
ax.barh(leaderboard['name'], leaderboard['rmse'], color=colors)
ax.set_xlabel('Test RMSE (lower is better)')
ax.set_title('Improvement Stack: each step vs baseline')
ax.invert_yaxis()
for i, v in enumerate(leaderboard['rmse']):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


### 10.1 诚实的复盘:不是所有「技巧」都有效

如果你认真看了上面的 leaderboard,你会注意到一些**反直觉**的现象——这些现象比「成功堆叠」更值得学:

| 现象 | 我们的结果 | 解读 |
| --- | --- | --- |
| `log1p` 几乎无改进 | RMSE 0.511 → 0.513(略变差) | 树模型对单调变换**天然不变**——它只看特征排序,不看具体值。所以 log1p 对线性模型很有用,对 GBM 通常几乎无效。 |
| **去掉截断样本反而变差** | RMSE 0.511 → 0.526 | 测试集**仍然包含**截断样本,但训练集少了它们,模型在高价区完全没见过样本,泛化更差。**双刃剑!** |
| 比率特征收益微小 | RMSE 0.511 → 0.510(0.2%) | 同样,树模型本来就能近似比率;特征工程对线性模型才是巨大收益。 |
| **海岸线距离明显有效** | RMSE 0.511 → 0.496(3.0%) | 这个特征是**真正全新的信息**——它结合了经纬度 + 加州地理常识,模型从原始特征学不出来。 |
| **现代 GBM 跨越式提升** | LightGBM 跑到 RMSE 0.434(15.1%) | **换模型比改特征收益大得多**——前提是你已经有了像样的特征集。 |
| 综合实验未必最强 | 综合 0.447 ≈ 单独 LightGBM 0.434 | 把会拖后腿的技巧(log1p + drop capped)叠加进去,反而拉低了最强模型。 |

#### 三条价值连城的工程经验

1. **「常识上应该有用」≠「实际有用」**——log1p 对线性模型是教科书操作,对树模型几乎无效。**永远做实验验证**,别凭经验下结论。
2. **训练集与测试集分布要保持一致**——技巧 2 翻车的根本原因是你「美化」了训练集,但测试集没变。永远问自己:**这个改动会改变训练 vs 测试的分布吗?**
3. **不要无脑叠加技巧**——每加一个改动都要单独评估;**只保留有正收益的改动**,放弃看似聪明但拖后腿的。

> 💡 如果这一节让你觉得「机器学习好像没那么神」——恭喜,你在用工程师的眼光看问题了。
> **真正神奇的是默认参数的 LightGBM**,而不是你能想到的所有「优化」。


## 11. 总结与下一步学习路径

### 你学到了什么

| 章节 | 技能 |
| --- | --- |
| §2-3 | 决策树、随机森林、梯度提升的**直觉**(不是 API,而是底层思想) |
| §5 | **目标变换**:对偏分布取 log,记得反变换;但树模型对此天然不敏感 |
| §6 | **数据清洗的陷阱**:训练集去掉截断样本但测试集没改 → 反而变差 |
| §7 | **特征工程**:比率特征对线性模型大有用,对树模型增益小 |
| §8 | **领域知识是金矿**:海岸线距离这种特征 = 全新信息 |
| §9 | **工业级 GBM 库**:XGBoost / LightGBM / CatBoost 的差异与选型 |
| §10.1 | **不是所有技巧都有效**——验证比直觉重要 |

### 工程经验

1. **先有基线,再改进** — 没有基线,所有「优化」都是空喊;
2. **一次只改一个变量** — 否则你不知道是哪个改进起了作用;
3. **特征工程 vs 模型选型** — 在加州数据上,**换 LightGBM 的收益(15%)远大于所有特征工程加起来(3-4%)**。
   但**前提**是你已经有像样的特征,否则再强的模型也救不了;
4. **领域知识 > 通用技巧** — log1p 是通用技巧但帮不上树模型;海岸线距离是领域知识但收益明显;
5. **要敢于扔掉看似聪明的改动** — 综合实验里 log1p + drop capped 拖了后腿,**不如不加**。

### 下一步学习路径

按从易到难推荐:

1. **`HistGradientBoostingRegressor`**(sklearn 原生):介于 sklearn GBR 和 LightGBM 之间,免依赖;
2. **Optuna 自动调参**:用贝叶斯搜索代替手动 GridSearch,通常能再榨 1-3% 的 RMSE;
3. **Stacking / Blending**:把 XGBoost、LightGBM、CatBoost 的预测加权融合;
4. **SHAP 值**:严肃的特征重要性 / 可解释性;
5. **表格数据上的现代 DL**:TabNet、FT-Transformer、SAINT(它们能不能打败 GBM 至今仍有争议)。

### 一个开放思考题

> 我们最好的模型把 RMSE 从 0.511 压到 0.434,改进约 15%。剩下的误差从哪儿来?
> 是模型不够强、特征还不够好,**还是数据本身就不够告诉你答案**(1990 年的普查特征不可能完全决定房价)?
> 这个问题没有标准答案——但思考它,你就开始像 ML 工程师那样思考问题了。
